# 15 — Per-target aggregation

Per-target means for every MD feature, plotted as a min-max-scaled heatmap (row = target, cell = mean of the 30 complexes). Shows which targets sit at the hard end of each axis (5HU9, 3I06) and which are easy (8ELC, 4L7G). That family context is what Act 4's ranking numbers ride on.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/15_per_target_aggregation_figK.png`.)_


> **Reader guide.** *Experiment A3:* per-target aggregation of the ~60 per-complex features.
>
> **Method:** per-target mean + min-max heatmap over ligands.
>
> **Reproducibility contract:** reads `data/raw/complex_analyses/`; per-target table to
> `data/derived/31_per_target_aggregation_data.csv`.

In [ ]:
# --- notebook preamble ---
NB_STEM = "31_per_target_aggregation"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 3. Per-target aggregation — the target-level baseline

Every downstream analysis has to be **within-target**. Targets differ in pocket geometry, protein size, ligand chemistry, and force-field parameterisation quality. `lig_drift = 2.5 Å` might be tight for target A (small, rigid pocket) and loose for target B (large pocket with a flexible loop).

The heatmap shows **min-max scaled means per target** so different scales don't dominate. Read across a row to spot which targets deviate on a given metric. The annotation is the raw mean across that target's 30 complexes.


In [ ]:

STABILITY_COLS = [
    'rmsd_bb_mean_A', 'rmsd_as_bb_mean_A', 'protein_rg_mean_A',
    'as_ca_rmsf_mean_A', 'as_ca_rmsf_max_A',
    'lig_drift_mean_A', 'lig_com_disp_max_A', 'lig_escape_frac',
    'lig_internal_rmsd_mean_A', 'lig_rmsf_mean_A',
    'lig_buried_sasa_mean_A2', 'vdw_contacts_mean',
    'n_hb_mean', 'hb_persistence_frac',
    'salt_bridges_lp_mean', 'ifp_tanimoto_median_vs_ref',
    'lig_binding_modes_2A', 'lig_orient_autocorr_mean',
    'coulomb_mean_arb', 'lig_dipole_mean_eA',
]

agg = df.groupby('target')[STABILITY_COLS].mean().round(3)
scaled = (agg - agg.min()) / (agg.max() - agg.min() + 1e-12)

fig, ax = plt.subplots(figsize=(12, 6.5))
im = ax.imshow(scaled.T.values, cmap='RdYlBu_r', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(agg.index))); ax.set_xticklabels(agg.index, fontsize=10)
ax.set_yticks(range(len(STABILITY_COLS))); ax.set_yticklabels(STABILITY_COLS, fontsize=8)
for i in range(agg.T.shape[0]):
    for j in range(agg.T.shape[1]):
        v = agg.T.values[i, j]
        col = WHITE if scaled.T.values[i,j] > 0.6 or scaled.T.values[i,j] < 0.15 else NAVY
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7, color=col)
cbar = plt.colorbar(im, ax=ax, label='min-max scaled', shrink=0.7)
cbar.ax.yaxis.label.set_color(NAVY)
ax.set_title('Per-target mean of each feature  (colour = min-max scaled; text = raw mean)')

**What to read first.**
- **`lig_drift_mean_A`** — a very high row mean means the target is dominated by escape/unstable poses. Cross-check against the docking-box scan (§ initial-frame report). High initial burial plus high MD drift usually means the docking box picked the wrong pocket — right cavity, wrong site.
- **`hb_persistence_frac`** — targets where the median complex holds > 0.7 usually have a clear HB pharmacophore (hinge Asp/Glu). Low targets are hydrophobic-dominant pockets.
- **`protein_rg_mean_A`** — target-size proxy (small ≈ 14 Å, large ≈ 25 Å). Not a discriminator, but sets scale expectations for the other features.
- **`active_site_formal_charge`** (see § 1 table) — negative pocket charge should favour cationic ligands. That shows up as elevated `salt_bridges_lp_mean` for the actives.


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
